In [3]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle as wrangle
import pandas as pd
from tz_pypsa.model import Model
import os
import glob
from pathlib import Path
import gc
from tz_pypsa.wrangle import get_ci_cost_summary

In [4]:
def process_and_save_networks_by_directory(
        base_path: str,
        output_base_path: str,
        pattern: str = "JPN_P1_JPN*",
        network_dir: str = "solved_networks",
        pypsa_run_id: str = 'Unspecified'
    ) -> None:
    """
    Process all .nc files in each solved_networks directory one at a time and
    save results to CSV immediately after processing each directory.

    Parameters
    ----------
    base_path : str
        Base directory path containing the run folders
    output_base_path : str
        Base path where to save the output CSV files
    pattern : str, optional
        Pattern to match subdirectories, defaults to "JPN_P1_JPN*"
    network_dir : str, optional
        Name of directory containing network files, defaults to "solved_networks"
    """

    # Create output directories
    yearly_path = os.path.join(output_base_path, "yearly")
    os.makedirs(yearly_path, exist_ok=True)

    unit_cost_dfs = []
    # Process each directory one at a time
    for dir_path in glob.glob(os.path.join(base_path, pattern)):
        dir_name = os.path.basename(dir_path)
        network_path = os.path.join(dir_path, network_dir)
        
        if not os.path.exists(network_path):
            print(f"Skipping {dir_name}: {network_dir} directory not found")
            continue
            
        # Find all .nc files in the solved_networks directory
        nc_files = glob.glob(os.path.join(network_path, "*.nc"))
        
        if not nc_files:
            print(f"No .nc files found in {network_path}")
            continue
            
        print(f"Processing {len(nc_files)} files in {dir_name}")
        
        # Process each .nc file and collect DataFrames
        yearly_dfs = []
        
        for nc_file in nc_files:
            try:
                # Load network
                network = pypsa.Network()
                network.import_from_netcdf(nc_file)
                
                # Get filename without extension for run_id
                scenario_id = Path(nc_file).stem
                
                yearly_df = wrangle.get_ci_unit_cost(
                    n=network
                )
                yearly_df['scenario_id'] = scenario_id
                yearly_df['pypsa_run_id'] = pypsa_run_id

                yearly_dfs.append(yearly_df)
                
                # Clean up to free memory
                del network
                gc.collect()
                
            except Exception as e:
                print(f"Error processing {nc_file}: {str(e)}")
                continue
        
        if yearly_dfs:
            yearly_output = pd.concat(yearly_dfs, ignore_index=True)
            print(f"Successfully processed and saved results for {dir_name}")
        else:
            print(f"No valid data processed for {dir_name}")

        unit_cost_dfs.append(yearly_output)

    unit_cost_output = pd.concat(unit_cost_dfs, ignore_index=True)
    unit_cost_output.to_csv(os.path.join(yearly_path, f"{pypsa_run_id}_unit_costs.csv"), index=False)

In [10]:
base_path = "/home/jy/tza-google-cfe/outputs/model_runs"
output_path = "/home/jy/tza-pypsa/csv_outputs"

process_and_save_networks_by_directory(
    base_path=base_path,
    output_base_path=output_path,
    pattern="TWN_P1_008",
    network_dir="solved_networks",
    pypsa_run_id="TWN_P1_008"
)

INFO:pypsa.io:Imported network hourly_matching_CFE95_2030.nc has buses, carriers, generators, links, loads, storage_units


Processing 8 files in TWN_P1_008


INFO:pypsa.io:Imported network hourly_matching_CFE70_2030.nc has buses, carriers, generators, links, loads, storage_units
INFO:pypsa.io:Imported network hourly_matching_CFE80_2030.nc has buses, carriers, generators, links, loads, storage_units
INFO:pypsa.io:Imported network hourly_matching_CFE100_2030.nc has buses, carriers, generators, links, loads, storage_units
INFO:pypsa.io:Imported network brownfield_2030.nc has buses, carriers, generators, links, loads, storage_units
INFO:pypsa.io:Imported network hourly_matching_CFE99_2030.nc has buses, carriers, generators, links, loads, storage_units
INFO:pypsa.io:Imported network hourly_matching_CFE90_2030.nc has buses, carriers, generators, links, loads, storage_units
INFO:pypsa.io:Imported network annual_matching_RES100_2030.nc has buses, carriers, generators, links, loads, storage_units


Successfully processed and saved results for TWN_P1_008
